In [1]:
import os
import pandas as pd
import boto3
from tqdm import tqdm
import datetime as dt

In [2]:
print(f'Last run date: {dt.datetime.today()}')

Last run date: 2024-03-07 21:15:31.011030


### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

Project: 20231010-gen-xii
Task: 08_retro_scoring
Subtask: 04_concat_results


### Get the names of the files in the s3 bucket

In [4]:
# init
cls_client = boto3.client('s3')

# prefix
str_prefix = '08_retro_scoring/03_step_function/parsed'

# list the files
dict_response = cls_client.list_objects_v2(Bucket=str_project, Prefix=str_prefix)

# get contents
list_dict_contents = dict_response['Contents']

# get the filenames
list_str_files = [dict_contents['Key'] for dict_contents in list_dict_contents]

# get only gzip
list_str_files = [str_file for str_file in list_str_files if '.gzip' in str_file]
print(f'There are {len(list_str_files)} parsed files:')
for a, str_file in enumerate(list_str_files):
    print(f'{a+1} - {str_file}')

There are 585 parsed files:
1 - 08_retro_scoring/03_step_function/parsed/X_raw_1.gzip
2 - 08_retro_scoring/03_step_function/parsed/X_raw_10.gzip
3 - 08_retro_scoring/03_step_function/parsed/X_raw_100.gzip
4 - 08_retro_scoring/03_step_function/parsed/X_raw_101.gzip
5 - 08_retro_scoring/03_step_function/parsed/X_raw_102.gzip
6 - 08_retro_scoring/03_step_function/parsed/X_raw_103.gzip
7 - 08_retro_scoring/03_step_function/parsed/X_raw_104.gzip
8 - 08_retro_scoring/03_step_function/parsed/X_raw_105.gzip
9 - 08_retro_scoring/03_step_function/parsed/X_raw_106.gzip
10 - 08_retro_scoring/03_step_function/parsed/X_raw_107.gzip
11 - 08_retro_scoring/03_step_function/parsed/X_raw_108.gzip
12 - 08_retro_scoring/03_step_function/parsed/X_raw_109.gzip
13 - 08_retro_scoring/03_step_function/parsed/X_raw_11.gzip
14 - 08_retro_scoring/03_step_function/parsed/X_raw_110.gzip
15 - 08_retro_scoring/03_step_function/parsed/X_raw_111.gzip
16 - 08_retro_scoring/03_step_function/parsed/X_raw_112.gzip
17 - 08_r

### Concatenate files

In [5]:
%%time

list_df = []
for str_file in tqdm(list_str_files):
    # read
    str_uri = f's3://{str_project}/{str_file}'
    df = pd.read_parquet(str_uri)
    # append
    list_df.append(df)
# concat
df = pd.concat(list_df)
# save memory
del list_df

# show
df

100%|██████████| 585/585 [04:02<00:00,  2.41it/s]


CPU times: user 2min 48s, sys: 25.8 s, total: 3min 13s
Wall time: 4min 18s


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,bigstatusid__app,strcity__app,strname__app,strzipcode__app,applicationdate__app,bitapproved__app,...,pti__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,bigAccountId,dtmFunded,BITDEBTOR,ln_was_empty__ln,tu_was_empty__tu
5514485__primary__20210120,0__0__20210120,5514485,NaN,1,14.0,Stanley,Virginia,22851,NaN,False,...,NaN,NaN,NaN,NaN,NaN,5514485,2021-01-25,1,NaN,NaN
5648765__primary__20210507,0__0__20210507,5648765,NaN,1,14.0,CUDAHY,Wisconsin,53110,NaN,False,...,NaN,NaN,NaN,NaN,NaN,5648765,2021-05-26,1,NaN,NaN
5648765__secondary__20210507,0__0__20210507,5648765,NaN,0,14.0,CUDAHY,Wisconsin,53110,NaN,False,...,NaN,NaN,NaN,NaN,NaN,5648765,2021-05-26,0,NaN,NaN
5678846__primary__20210611,0__0__20210611,5678846,NaN,1,14.0,WESTLAKE,Louisiana,70669,NaN,False,...,NaN,NaN,NaN,NaN,NaN,5678846,2021-07-01,1,NaN,NaN
5721519__7185736__20210728,5721519__7185736__20210728,5721519,7185736.0,1,NaN,GAITHERSBURG,Maryland,20877,NaN,True,...,NaN,NaN,NaN,NaN,NaN,5721519,2021-08-03,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7287114__primary__20231030,0__0__20231030,7287114,NaN,1,11.0,PHILADELPHIA,Pennsylvania,19131,NaN,False,...,NaN,NaN,NaN,NaN,NaN,7287114,2023-11-20,1,NaN,NaN
7287114__secondary__20231030,0__0__20231030,7287114,NaN,0,11.0,PHILADELPHIA,Pennsylvania,19131,NaN,False,...,NaN,NaN,NaN,NaN,NaN,7287114,2023-11-20,0,NaN,NaN
7354598__primary__20231117,0__0__20231117,7354598,NaN,1,9.0,GLENDALE,Arizona,85306,NaN,False,...,NaN,NaN,NaN,NaN,NaN,7354598,2023-11-29,1,NaN,NaN
7354598__secondary__20231117,0__0__20231117,7354598,NaN,0,9.0,GLENDALE,Arizona,85306,NaN,False,...,NaN,NaN,NaN,NaN,NaN,7354598,2023-11-29,0,NaN,NaN


### Write to s3

In [6]:
%%time

# convert to string
for col in df.columns:
    if df[col].dtype not in ['int64', 'float64']:
        df[col] = df[col].astype(str)

str_filename = 'df_X_raw.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 24.6 s, sys: 216 ms, total: 24.8 s
Wall time: 24.8 s
